# ENGRAMA V5 — Validación de recuperación en contexto largo (hasta 8192 tokens)

Este notebook valida en **GPU** lo que el sandbox de CPU no puede: recuperación
clave→valor exacta a **contextos enormes**. V5 usa **resonancia sináptica** sobre
la traza explícita — **sin atención, sin compresión**.

Objetivo: **> 85% de recall** a 8000+ tokens.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'engrama'], check=False)
# Si prefieres el código local del repo:
# subprocess.run([sys.executable,'-m','pip','install','-q','-e','/kaggle/input/engrama'], check=False)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## 1. Definición de la tarea de recuperación (escalable a cualquier longitud)


In [ ]:
import random, time, math
import torch, torch.nn.functional as F

VOCAB=64; KEY_LO,KEY_HI=20,35; VAL_LO,VAL_HI=40,55; BOS=0
N_KEYS=6                      # 6 pares clave-valor por muestra
CHANCE=1.0/(VAL_HI-VAL_LO+1)  # azar

def make_sample(rng, SEQ):
    header=list(range(1, 1+2*N_KEYS, 2))
    q0=max(header)+8
    queries=[q0 + i*((SEQ-q0-2)//N_KEYS) for i in range(N_KEYS)]
    keys=rng.sample(range(KEY_LO,KEY_HI+1),N_KEYS)
    vals=[rng.randint(VAL_LO,VAL_HI) for _ in range(N_KEYS)]
    vo=dict(zip(keys,vals))
    seq=[BOS]+[rng.randint(4,15) for _ in range(SEQ-1)]
    for slot,(k,v) in zip(header,zip(keys,vals)): seq[slot]=k; seq[slot+1]=v
    order=keys[:]; rng.shuffle(order); tg=[]
    for pos,k in zip(queries,order): seq[pos]=k; seq[pos+1]=vo[k]; tg.append((pos,vo[k]))
    return torch.tensor(seq),tg

def make_batch(bs,rng,SEQ):
    s=[make_sample(rng,SEQ) for _ in range(bs)]
    return torch.stack([x for x,_ in s]),[t for _,t in s]
print('chance =', f'{CHANCE:.1%}')


## 2. Construir el modelo V5


In [ ]:
from engrama import EngramaV5, EngramaV5Config

SEQ = 8192          # <-- contexto largo real
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = EngramaV5Config(
    vocab_size=VOCAB, d_model=256, num_layers=4, num_heads=8,
    context_length=SEQ, num_candidates=1, norm_type='layernorm',
    chunk_size=512,   # tiling causal: memoria acotada en 8k, idéntico al forward completo
)
model = EngramaV5(cfg).to(device)
print(model.describe())
print('params:', f'{model.num_parameters():,}')


## 3. Entrenamiento (batch pequeño por la longitud; usa AMP en GPU)


In [ ]:
BATCH=4; STEPS=4000; ANSWER_WEIGHT=10.0
opt=torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01, betas=(0.9,0.95))
scaler=torch.amp.GradScaler('cuda', enabled=(device=='cuda'))
rng=random.Random(7)

def evaluate(SEQ, samples=64):
    model.eval(); per=[0]*N_KEYS; ev=random.Random(999)
    with torch.no_grad():
        left=samples
        while left>0:
            b=min(BATCH,left); left-=b
            seqs,ans=make_batch(b,ev,SEQ)
            with torch.amp.autocast('cuda', enabled=(device=='cuda')):
                preds=model(seqs[:,:-1].to(device)).argmax(-1)
            for r,a in enumerate(ans):
                for qi,(pos,v) in enumerate(a): per[qi]+=int(preds[r,pos].item()==v)
    model.train(); return sum(per)/(samples*N_KEYS), [p/samples for p in per]

model.train(); t0=time.time()
for step in range(1,STEPS+1):
    seqs,ans=make_batch(BATCH,rng,SEQ)
    with torch.amp.autocast('cuda', enabled=(device=='cuda')):
        logits=model(seqs[:,:-1].to(device)); tgt=seqs[:,1:].to(device)
        raw=F.cross_entropy(logits.reshape(-1,VOCAB),tgt.reshape(-1),reduction='none').view(BATCH,-1)
        w=torch.ones_like(raw)
        for r,a in enumerate(ans):
            for pos,_ in a: w[r,pos]=ANSWER_WEIGHT
        loss=(raw*w).mean()
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(opt); scaler.update()
    if step % (STEPS//10) == 0:
        acc,_=evaluate(SEQ)
        print(f'step {step:5d}  loss {loss.item():.3f}  recall {acc:.1%}  ({(time.time()-t0)/step*1000:.0f} ms/step)')


## 4. Resultado final por distancia de recuperación


In [ ]:
acc, per = evaluate(SEQ, samples=128)
print(f'RECALL GLOBAL a SEQ={SEQ}: {acc:.1%}  (azar {CHANCE:.1%})')
print('por consulta (distancia creciente):', [f'{p:.1%}' for p in per])
assert acc > 0.85, 'objetivo >85% no alcanzado — sube STEPS o capacidad'


## 5. Garantías: invarianza causal + memoria lineal de generación


In [ ]:
# Invarianza causal en una secuencia corta
model.eval()
ids=torch.randint(0,VOCAB,(1,64)).to(device)
with torch.no_grad():
    par=model.forward(ids)
    cache=model.new_cache(); inc=torch.zeros_like(par)
    for t in range(64): inc[0,t]=model._step_logits(ids[0,t:t+1],cache)[0]
print('invarianza causal |Δ|:', (par-inc).abs().max().item())

# Memoria de la caché == O(N) (bytes/token constante)
for N in [512,1024,2048,4096]:
    c=model.new_cache()
    with torch.no_grad():
        for t in range(N): model._step_logits(torch.tensor([t%VOCAB]).to(device),c)
    print(f'N={N:5d}  bytes/token={c.memory_bytes()//N}')


## Conclusión

Si `RECALL GLOBAL` supera 85% a 8192 tokens, ENGRAMA V5 cumple el requisito de
recuperación en contextos enormes **sin atención y sin compresión**, con memoria
de generación estrictamente lineal (bytes/token constante) e invarianza causal
exacta. Ajusta `STEPS`, `d_model`, `num_layers` si necesitas más margen.
